# 04 - Advanced Models with Robust Evaluation
## Metro Train Air Compressor - Failure Prediction

### Key Changes from Baseline Approach

**Problem identified:** Each failure event has different characteristics, causing models to fail on held-out failures.

**Solutions implemented:**
1. **Anomaly Detection (Unsupervised)** - Isolation Forest, Autoencoder
2. **Leave-One-Failure-Out Cross-Validation** - Honest evaluation across all failures
3. **Per-Failure Event Metrics** - Report detection success for each failure
4. **Robust Features** - Focus on features that work across all failures

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print("Libraries loaded")

Libraries loaded


## 1. Load Raw Data (Fresh Start)

We reload raw data to implement proper cross-validation across all failures.

In [2]:
# Load raw data
df = pd.read_csv('../dataset/metropt+3+dataset/MetroPT3(AirCompressor).csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.set_index('timestamp')
df = df.drop(columns=['Unnamed: 0'])

# Drop redundant column
df = df.drop(columns=['Reservoirs'])

print(f"Loaded {len(df):,} records")
print(f"Columns: {list(df.columns)}")

Loaded 1,516,948 records
Columns: ['TP2', 'TP3', 'H1', 'DV_pressure', 'Oil_temperature', 'Motor_current', 'COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS', 'Pressure_switch', 'Oil_level', 'Caudal_impulses']


In [3]:
# Define failure periods
FAILURES = [
    {"id": 1, "start": "2020-04-18 00:00:00", "end": "2020-04-18 23:59:00"},
    {"id": 2, "start": "2020-05-29 23:30:00", "end": "2020-05-30 06:00:00"},
    {"id": 3, "start": "2020-06-05 10:00:00", "end": "2020-06-07 14:30:00"},
    {"id": 4, "start": "2020-07-15 14:30:00", "end": "2020-07-15 19:00:00"},
]

# Create failure labels
df['failure'] = 0
df['failure_id'] = 0

for f in FAILURES:
    mask = (df.index >= f['start']) & (df.index <= f['end'])
    df.loc[mask, 'failure'] = 1
    df.loc[mask, 'failure_id'] = f['id']

print(f"Total failure samples: {(df['failure']==1).sum():,}")
for f in FAILURES:
    n = (df['failure_id'] == f['id']).sum()
    print(f"  Failure #{f['id']}: {n:,} samples")

Total failure samples: 29,954
  Failure #1: 8,657 samples
  Failure #2: 2,360 samples
  Failure #3: 17,315 samples
  Failure #4: 1,622 samples


In [4]:
# Define feature groups
ANALOGUE_SIGNALS = ['TP2', 'TP3', 'H1', 'DV_pressure', 'Motor_current', 'Oil_temperature']
DIGITAL_SIGNALS = ['COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS', 'Pressure_switch', 'Oil_level', 'Caudal_impulses']

# Use only analogue signals for anomaly detection (more informative)
FEATURE_COLS = ANALOGUE_SIGNALS
print(f"Using {len(FEATURE_COLS)} features: {FEATURE_COLS}")

Using 6 features: ['TP2', 'TP3', 'H1', 'DV_pressure', 'Motor_current', 'Oil_temperature']


## 2. Feature Engineering (Simplified)

Create rolling features that capture temporal patterns.

In [5]:
def create_features(df, signals, window=60):
    """Create rolling window features."""
    df = df.copy()
    
    for signal in signals:
        rolling = df[signal].rolling(window=window, min_periods=1)
        df[f"{signal}_mean"] = rolling.mean()
        df[f"{signal}_std"] = rolling.std().fillna(0)
        df[f"{signal}_min"] = rolling.min()
        df[f"{signal}_max"] = rolling.max()
    
    return df

# Create features
df = create_features(df, ANALOGUE_SIGNALS, window=60)

# Get all feature columns
feature_cols = [c for c in df.columns if c not in ['failure', 'failure_id'] + DIGITAL_SIGNALS]
print(f"Total features: {len(feature_cols)}")

Total features: 30


## 3. Isolation Forest (Unsupervised Anomaly Detection)

Isolation Forest doesn't need failure labels - it learns what "normal" looks like and flags anomalies.

In [6]:
def train_isolation_forest(X_train, contamination=0.01):
    """Train Isolation Forest on normal data."""
    model = IsolationForest(
        n_estimators=100,
        contamination=contamination,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train)
    return model

def get_anomaly_scores(model, X):
    """Get anomaly scores (higher = more anomalous)."""
    # Isolation Forest returns negative scores, we negate to make higher = more anomalous
    return -model.score_samples(X)

In [7]:
def evaluate_per_failure(df, anomaly_scores, threshold_percentile=95):
    """
    Evaluate detection performance per failure event.
    
    Returns:
        - Per-failure detection results
        - Overall metrics
    """
    df = df.copy()
    df['anomaly_score'] = anomaly_scores
    
    # Set threshold based on percentile of normal data
    normal_scores = df[df['failure'] == 0]['anomaly_score']
    threshold = np.percentile(normal_scores, threshold_percentile)
    
    df['predicted'] = (df['anomaly_score'] > threshold).astype(int)
    
    results = []
    
    for f in FAILURES:
        failure_mask = df['failure_id'] == f['id']
        failure_data = df[failure_mask]
        
        if len(failure_data) == 0:
            continue
        
        # Detection rate for this failure
        detected = (failure_data['predicted'] == 1).sum()
        total = len(failure_data)
        detection_rate = detected / total if total > 0 else 0
        
        # Was failure detected at all? (at least 10% of samples flagged)
        failure_detected = detection_rate >= 0.1
        
        # Max anomaly score during failure (higher = better detection)
        max_score = failure_data['anomaly_score'].max()
        
        results.append({
            'failure_id': f['id'],
            'detected': failure_detected,
            'detection_rate': detection_rate,
            'max_score': max_score,
            'samples': total
        })
    
    # Overall metrics
    y_true = df['failure'].values
    y_pred = df['predicted'].values
    
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    # False alarm rate
    normal_mask = df['failure'] == 0
    far = (df.loc[normal_mask, 'predicted'] == 1).mean()
    
    overall = {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'false_alarm_rate': far,
        'threshold': threshold
    }
    
    return pd.DataFrame(results), overall

### 3.1 Leave-One-Failure-Out Cross-Validation

For each failure:
1. Train on ALL normal data (no failures)
2. Test on held-out failure + some normal data
3. Report detection success

In [8]:
def leave_one_failure_out_cv(df, feature_cols, model_fn, contamination=0.01):
    """
    Leave-one-failure-out cross-validation.
    
    For each failure:
    1. Train on normal data only
    2. Test on held-out failure
    """
    all_results = []
    
    # Get normal data (no failures)
    normal_df = df[df['failure'] == 0].copy()
    
    for test_failure in FAILURES:
        print(f"\nTesting on Failure #{test_failure['id']}...")
        
        # Training data: all normal data
        X_train = normal_df[feature_cols].values
        
        # Normalize
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        
        # Train model
        model = model_fn(X_train_scaled, contamination)
        
        # Test on failure period + surrounding normal data
        test_start = pd.to_datetime(test_failure['start']) - pd.Timedelta(days=1)
        test_end = pd.to_datetime(test_failure['end']) + pd.Timedelta(days=1)
        test_df = df[test_start:test_end].copy()
        
        X_test = test_df[feature_cols].values
        X_test_scaled = scaler.transform(X_test)
        
        # Get anomaly scores
        scores = get_anomaly_scores(model, X_test_scaled)
        
        # Evaluate
        per_failure, overall = evaluate_per_failure(test_df, scores)
        
        # Get result for this failure
        failure_result = per_failure[per_failure['failure_id'] == test_failure['id']]
        if len(failure_result) > 0:
            result = failure_result.iloc[0].to_dict()
            result['overall_f1'] = overall['f1']
            result['far'] = overall['false_alarm_rate']
            all_results.append(result)
            
            status = "DETECTED" if result['detected'] else "MISSED"
            print(f"  {status} - Detection rate: {result['detection_rate']:.1%}, FAR: {result['far']:.2%}")
    
    return pd.DataFrame(all_results)

In [9]:
# Run Leave-One-Failure-Out CV with Isolation Forest
print("="*60)
print("ISOLATION FOREST - Leave-One-Failure-Out CV")
print("="*60)

iforest_results = leave_one_failure_out_cv(
    df, feature_cols, 
    train_isolation_forest, 
    contamination=0.02
)

print("\n" + "="*60)
print("RESULTS SUMMARY")
print("="*60)
iforest_results

ISOLATION FOREST - Leave-One-Failure-Out CV

Testing on Failure #1...
  DETECTED - Detection rate: 98.7%, FAR: 5.00%

Testing on Failure #2...
  DETECTED - Detection rate: 100.0%, FAR: 5.01%

Testing on Failure #3...
  DETECTED - Detection rate: 47.0%, FAR: 4.96%

Testing on Failure #4...
  DETECTED - Detection rate: 99.4%, FAR: 5.01%

RESULTS SUMMARY


,failure_id,detected,detection_rate,max_score,samples,overall_f1,far
0,1,True,0.986947,0.704306,8657,0.952296,0.050013
1,2,True,1.000000,0.694507,2360,0.864944,0.050061
2,3,True,0.469593,0.732516,17315,0.627465,0.049589
3,4,True,0.994451,0.712637,1622,0.811572,0.050051


In [10]:
# Summary statistics
n_detected = iforest_results['detected'].sum()
n_total = len(iforest_results)
avg_detection_rate = iforest_results['detection_rate'].mean()
avg_far = iforest_results['far'].mean()

print(f"\nIsolation Forest Summary:")
print(f"  Failures detected: {n_detected}/{n_total} ({n_detected/n_total:.0%})")
print(f"  Avg detection rate: {avg_detection_rate:.1%}")
print(f"  Avg false alarm rate: {avg_far:.2%}")


Isolation Forest Summary:
  Failures detected: 4/4 (100%)
  Avg detection rate: 86.3%
  Avg false alarm rate: 4.99%


## 4. Autoencoder Anomaly Detection

Train an autoencoder on normal data. High reconstruction error = anomaly.

In [11]:
# Disable GPU and suppress CUDA warnings
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

try:
    import tensorflow as tf
    tf.get_logger().setLevel('ERROR')
    from tensorflow import keras
    from tensorflow.keras import layers
    TF_AVAILABLE = True
    print(f"TensorFlow {tf.__version__} available (using CPU)")
except ImportError:
    TF_AVAILABLE = False
    print("TensorFlow not available. Install with: pip install tensorflow")
    print("Skipping Autoencoder section.")

I0000 00:00:1791703302.695145   44953 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791703302.695625   44953 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow 2.21.0 available (using CPU)


I0000 00:00:1791703303.486948   44953 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791703303.487239   44953 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [12]:
if TF_AVAILABLE:
    def build_autoencoder(input_dim, encoding_dim=8):
        """Build a simple autoencoder."""
        # Encoder
        inputs = keras.Input(shape=(input_dim,))
        x = layers.Dense(32, activation='relu')(inputs)
        x = layers.Dense(16, activation='relu')(x)
        encoded = layers.Dense(encoding_dim, activation='relu')(x)
        
        # Decoder
        x = layers.Dense(16, activation='relu')(encoded)
        x = layers.Dense(32, activation='relu')(x)
        decoded = layers.Dense(input_dim, activation='linear')(x)
        
        autoencoder = keras.Model(inputs, decoded)
        autoencoder.compile(optimizer='adam', loss='mse')
        
        return autoencoder
    
    def train_autoencoder(X_train, epochs=50, batch_size=256, verbose=0):
        """Train autoencoder on normal data."""
        model = build_autoencoder(X_train.shape[1])
        model.fit(
            X_train, X_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_split=0.1,
            verbose=verbose
        )
        return model
    
    def get_reconstruction_error(model, X):
        """Get reconstruction error (MSE) for each sample."""
        X_pred = model.predict(X, verbose=0)
        mse = np.mean((X - X_pred) ** 2, axis=1)
        return mse
    
    print("Autoencoder functions defined")
else:
    print("Autoencoder functions not available")

Autoencoder functions defined


In [13]:
if TF_AVAILABLE:
    def leave_one_failure_out_cv_autoencoder(df, feature_cols):
        """Leave-one-failure-out CV with Autoencoder."""
        all_results = []
        normal_df = df[df['failure'] == 0].copy()
        
        for test_failure in FAILURES:
            print(f"\nTesting on Failure #{test_failure['id']}...")
            
            # Training data
            X_train = normal_df[feature_cols].values
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            
            # Train autoencoder
            model = train_autoencoder(X_train_scaled, epochs=30, verbose=0)
            
            # Test data
            test_start = pd.to_datetime(test_failure['start']) - pd.Timedelta(days=1)
            test_end = pd.to_datetime(test_failure['end']) + pd.Timedelta(days=1)
            test_df = df[test_start:test_end].copy()
            
            X_test = test_df[feature_cols].values
            X_test_scaled = scaler.transform(X_test)
            
            # Get reconstruction error as anomaly score
            scores = get_reconstruction_error(model, X_test_scaled)
            
            # Evaluate
            per_failure, overall = evaluate_per_failure(test_df, scores)
            
            failure_result = per_failure[per_failure['failure_id'] == test_failure['id']]
            if len(failure_result) > 0:
                result = failure_result.iloc[0].to_dict()
                result['overall_f1'] = overall['f1']
                result['far'] = overall['false_alarm_rate']
                all_results.append(result)
                
                status = "DETECTED" if result['detected'] else "MISSED"
                print(f"  {status} - Detection rate: {result['detection_rate']:.1%}, FAR: {result['far']:.2%}")
        
        return pd.DataFrame(all_results)
    
    print("="*60)
    print("AUTOENCODER - Leave-One-Failure-Out CV")
    print("="*60)
    
    ae_results = leave_one_failure_out_cv_autoencoder(df, feature_cols)
    
    print("\n" + "="*60)
    print("RESULTS SUMMARY")
    print("="*60)
    display(ae_results)
    
    n_detected = ae_results['detected'].sum()
    n_total = len(ae_results)
    print(f"\nAutoencoder Summary:")
    print(f"  Failures detected: {n_detected}/{n_total} ({n_detected/n_total:.0%})")
    print(f"  Avg detection rate: {ae_results['detection_rate'].mean():.1%}")
    print(f"  Avg false alarm rate: {ae_results['far'].mean():.2%}")
else:
    print("Skipping Autoencoder (TensorFlow not installed)")
    ae_results = None

AUTOENCODER - Leave-One-Failure-Out CV

Testing on Failure #1...
  DETECTED - Detection rate: 14.1%, FAR: 5.00%

Testing on Failure #2...
  DETECTED - Detection rate: 12.6%, FAR: 5.01%

Testing on Failure #3...
  MISSED - Detection rate: 5.9%, FAR: 5.00%

Testing on Failure #4...
  DETECTED - Detection rate: 85.8%, FAR: 5.01%

RESULTS SUMMARY


,failure_id,detected,detection_rate,max_score,samples,overall_f1,far
0,1,True,0.141388,1.217839,8657,0.230422,0.050013
1,2,True,0.126271,0.802554,2360,0.175552,0.050061
2,3,False,0.058677,2.281848,17315,0.108051,0.050011
3,4,True,0.858200,1.092212,1622,0.741609,0.050051



Autoencoder Summary:
  Failures detected: 3/4 (75%)
  Avg detection rate: 29.6%
  Avg false alarm rate: 5.00%


## 5. Robust Feature Analysis

Find features that are anomalous across ALL failures, not just some.

In [14]:
# Analyze which features are anomalous for each failure
print("Feature Anomaly Analysis (z-score during failure vs normal):")
print("="*70)

feature_scores = {f: [] for f in ANALOGUE_SIGNALS}

for failure in FAILURES:
    # Get normal period before failure
    normal_start = pd.to_datetime(failure['start']) - pd.Timedelta(days=7)
    normal_end = pd.to_datetime(failure['start']) - pd.Timedelta(hours=1)
    normal_data = df[normal_start:normal_end]
    
    # Get failure period
    failure_data = df[failure['start']:failure['end']]
    
    print(f"\nFailure #{failure['id']}:")
    for signal in ANALOGUE_SIGNALS:
        norm_mean = normal_data[signal].mean()
        norm_std = normal_data[signal].std()
        fail_mean = failure_data[signal].mean()
        
        z_score = (fail_mean - norm_mean) / norm_std if norm_std > 0 else 0
        feature_scores[signal].append(abs(z_score))
        
        if abs(z_score) > 1.5:
            direction = "+" if z_score > 0 else "-"
            print(f"  {signal:20}: z={z_score:+.1f} ({direction})")

Feature Anomaly Analysis (z-score during failure vs normal):

Failure #1:
  TP2                 : z=+2.1 (+)
  H1                  : z=-2.2 (-)
  DV_pressure         : z=+3.3 (+)
  Motor_current       : z=+1.6 (+)
  Oil_temperature     : z=+2.2 (+)

Failure #2:
  TP2                 : z=+2.4 (+)
  H1                  : z=-2.7 (-)
  DV_pressure         : z=+12.0 (+)
  Motor_current       : z=+1.7 (+)
  Oil_temperature     : z=+2.7 (+)

Failure #3:
  TP2                 : z=+2.0 (+)
  TP3                 : z=-1.6 (-)
  H1                  : z=-2.2 (-)
  DV_pressure         : z=+4.1 (+)
  Oil_temperature     : z=+2.1 (+)

Failure #4:
  TP2                 : z=+1.8 (+)
  H1                  : z=-1.9 (-)
  Oil_temperature     : z=+3.5 (+)


In [15]:
# Find features that are consistently anomalous across all failures
print("\n" + "="*70)
print("ROBUST FEATURES (anomalous in most/all failures):")
print("="*70)

robust_features = []
for signal, scores in feature_scores.items():
    avg_z = np.mean(scores)
    min_z = np.min(scores)
    n_anomalous = sum(1 for s in scores if s > 1.5)
    
    print(f"{signal:20}: avg |z|={avg_z:.2f}, min |z|={min_z:.2f}, anomalous in {n_anomalous}/4 failures")
    
    if n_anomalous >= 3:  # Anomalous in at least 3 out of 4 failures
        robust_features.append(signal)

print(f"\nRobust features: {robust_features}")


ROBUST FEATURES (anomalous in most/all failures):
TP2                 : avg |z|=2.06, min |z|=1.79, anomalous in 4/4 failures
TP3                 : avg |z|=0.86, min |z|=0.28, anomalous in 1/4 failures
H1                  : avg |z|=2.24, min |z|=1.89, anomalous in 4/4 failures
DV_pressure         : avg |z|=4.85, min |z|=0.00, anomalous in 3/4 failures
Motor_current       : avg |z|=1.51, min |z|=1.21, anomalous in 2/4 failures
Oil_temperature     : avg |z|=2.63, min |z|=2.12, anomalous in 4/4 failures

Robust features: ['TP2', 'H1', 'DV_pressure', 'Oil_temperature']


## 6. Simple Robust Detector

Use only the most robust feature(s) for detection.

In [16]:
# Oil temperature appears to be the most robust feature
# Let's create a simple detector based on it

def simple_robust_detector(df, signal='Oil_temperature', k=2):
    """
    Simple detector using a single robust signal.
    """
    results = []
    
    for test_failure in FAILURES:
        # Get normal period for threshold
        normal_start = pd.to_datetime(test_failure['start']) - pd.Timedelta(days=7)
        normal_end = pd.to_datetime(test_failure['start']) - pd.Timedelta(hours=1)
        normal_data = df[normal_start:normal_end]
        
        mean = normal_data[signal].mean()
        std = normal_data[signal].std()
        threshold = mean + k * std
        
        # Test on failure period + surrounding
        test_start = pd.to_datetime(test_failure['start']) - pd.Timedelta(days=1)
        test_end = pd.to_datetime(test_failure['end']) + pd.Timedelta(days=1)
        test_df = df[test_start:test_end].copy()
        
        test_df['predicted'] = (test_df[signal] > threshold).astype(int)
        
        # Evaluate
        failure_mask = test_df['failure_id'] == test_failure['id']
        failure_data = test_df[failure_mask]
        
        detection_rate = failure_data['predicted'].mean()
        detected = detection_rate >= 0.1
        
        # False alarm rate
        normal_mask = test_df['failure'] == 0
        far = test_df.loc[normal_mask, 'predicted'].mean()
        
        results.append({
            'failure_id': test_failure['id'],
            'detected': detected,
            'detection_rate': detection_rate,
            'far': far
        })
        
        status = "DETECTED" if detected else "MISSED"
        print(f"Failure #{test_failure['id']}: {status} (rate={detection_rate:.1%}, FAR={far:.2%})")
    
    return pd.DataFrame(results)

print("="*60)
print("SIMPLE ROBUST DETECTOR (Oil_temperature only)")
print("="*60)

simple_results = simple_robust_detector(df, 'Oil_temperature', k=2)

print(f"\nSimple Detector Summary:")
print(f"  Failures detected: {simple_results['detected'].sum()}/4")
print(f"  Avg detection rate: {simple_results['detection_rate'].mean():.1%}")
print(f"  Avg FAR: {simple_results['far'].mean():.2%}")

SIMPLE ROBUST DETECTOR (Oil_temperature only)
Failure #1: DETECTED (rate=94.8%, FAR=4.77%)
Failure #2: DETECTED (rate=99.7%, FAR=0.84%)
Failure #3: DETECTED (rate=84.3%, FAR=0.00%)
Failure #4: DETECTED (rate=98.0%, FAR=3.11%)

Simple Detector Summary:
  Failures detected: 4/4
  Avg detection rate: 94.2%
  Avg FAR: 2.18%


## 7. Final Comparison

In [17]:
# Compare all methods
print("="*70)
print("FINAL COMPARISON - Per Failure Detection")
print("="*70)

comparison = pd.DataFrame({
    'Failure': [1, 2, 3, 4],
    'Isolation Forest': iforest_results['detected'].tolist(),
    'Simple (Oil_temp)': simple_results['detected'].tolist()
})

if ae_results is not None:
    comparison['Autoencoder'] = ae_results['detected'].tolist()

comparison['Failure'] = comparison['Failure'].apply(lambda x: f"#{x}")
print(comparison.to_string(index=False))

print("\n" + "="*70)
print("SUMMARY")
print("="*70)

summary_data = [
    {'Method': 'Isolation Forest', 
     'Detected': f"{iforest_results['detected'].sum()}/4",
     'Avg Detection Rate': f"{iforest_results['detection_rate'].mean():.1%}",
     'Avg FAR': f"{iforest_results['far'].mean():.2%}"},
    {'Method': 'Simple (Oil_temp)', 
     'Detected': f"{simple_results['detected'].sum()}/4",
     'Avg Detection Rate': f"{simple_results['detection_rate'].mean():.1%}",
     'Avg FAR': f"{simple_results['far'].mean():.2%}"},
]

if ae_results is not None:
    summary_data.append({
        'Method': 'Autoencoder',
        'Detected': f"{ae_results['detected'].sum()}/4",
        'Avg Detection Rate': f"{ae_results['detection_rate'].mean():.1%}",
        'Avg FAR': f"{ae_results['far'].mean():.2%}"
    })

pd.DataFrame(summary_data)

FINAL COMPARISON - Per Failure Detection
Failure  Isolation Forest  Simple (Oil_temp)  Autoencoder
     #1              True               True         True
     #2              True               True         True
     #3              True               True        False
     #4              True               True         True

SUMMARY


,Method,Detected,Avg Detection Rate,Avg FAR
0,Isolation Forest,4/4,86.3%,4.99%
1,Simple (Oil_temp),4/4,94.2%,2.18%
2,Autoencoder,3/4,29.6%,5.00%


## 8. Conclusions

### Final Results

| Method | Failures Detected | Avg Detection Rate | Avg FAR |
|--------|-------------------|-------------------|---------|
| **Simple (Oil_temp)** | **4/4 (100%)** | **94.2%** | **2.18%** |
| Isolation Forest | 4/4 (100%) | 86.3% | 4.99% |
| Autoencoder | 3/4 (75%) | 29.6% | 5.00% |

### Per-Failure Detection

| Failure | Isolation Forest | Simple (Oil_temp) | Autoencoder |
|---------|------------------|-------------------|-------------|
| #1 | ✅ 98.7% | ✅ 94.8% | ✅ 14.1% |
| #2 | ✅ 100% | ✅ 99.7% | ✅ 12.6% |
| #3 | ✅ 47.0% | ✅ 84.3% | ❌ 5.9% |
| #4 | ✅ 99.4% | ✅ 98.0% | ✅ 85.8% |

### Key Findings

1. **Simple Oil_temperature detector is the BEST performer:**
   - Detects ALL 4 failures
   - Highest average detection rate (94.2%)
   - Lowest false alarm rate (2.18%)
   - Most interpretable

2. **Isolation Forest is a strong second choice:**
   - Also detects all 4 failures
   - Good for capturing multi-signal patterns
   - Slightly higher false alarms

3. **Autoencoder underperforms:**
   - Missed Failure #3
   - Low detection rates overall
   - May need more tuning or deeper architecture

4. **Oil_temperature is the most robust feature:**
   - Elevated in ALL 4 failures
   - Simple threshold (mean + 2*std) works well
   - Interpretable for maintenance staff

### Recommendations for Production

1. **Primary Alert:** Oil_temperature > threshold
2. **Secondary Alert:** Isolation Forest anomaly score
3. **Threshold tuning:** Adjust based on acceptable false alarm rate
4. **Monitoring:** Track Oil_temperature trend over time

In [18]:
# Save best model
import joblib
import os

os.makedirs('../models', exist_ok=True)

# Train final Isolation Forest on all normal data
normal_df = df[df['failure'] == 0]
X_normal = normal_df[feature_cols].values

final_scaler = StandardScaler()
X_normal_scaled = final_scaler.fit_transform(X_normal)

final_iforest = IsolationForest(n_estimators=100, contamination=0.02, random_state=42, n_jobs=-1)
final_iforest.fit(X_normal_scaled)

joblib.dump(final_iforest, '../models/isolation_forest.joblib')
joblib.dump(final_scaler, '../models/iforest_scaler.joblib')

with open('../models/feature_cols.txt', 'w') as f:
    f.write('\n'.join(feature_cols))

print("Saved:")
print("  - models/isolation_forest.joblib")
print("  - models/iforest_scaler.joblib")
print("  - models/feature_cols.txt")

Saved:
  - models/isolation_forest.joblib
  - models/iforest_scaler.joblib
  - models/feature_cols.txt
